# API Functions → Binary ML Dataset

Downloads `API Functions.csv` directly from a public S3 URL, then flattens it into a binary matrix suitable for machine learning:
- **Rows**: one per SHA256 sample
- **API columns**: 1 if that API function is present in the sample, 0 otherwise
- **Label columns**: one-hot encoded malware type (Backdoor, Downloader, Generic Malware, Ransomware, Spyware, …)

No file upload or Google Drive required — just run all cells in order.

## Step 1 — Imports

In [ ]:
import io
import os
import requests
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

print(f"pandas {pd.__version__}")

## Step 2 — Download CSV from URL

In [ ]:
CSV_URL     = "https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/a59c473d-9076-422a-a8de-952a35591757"
LOCAL_CSV   = "/content/API_Functions.csv"
OUTPUT_FILE = "/content/API_Functions_flat.csv"

if os.path.exists(LOCAL_CSV):
    print(f"Already cached: {LOCAL_CSV}")
else:
    print("Downloading API Functions.csv …")
    with requests.get(CSV_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(LOCAL_CSV, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {LOCAL_CSV}  ({os.path.getsize(LOCAL_CSV) / 1e6:.1f} MB)")

## Step 3 — Load CSV

In [ ]:
print("Reading CSV …")

df = pd.read_csv(LOCAL_CSV, header=None, index_col=0, low_memory=False)
df.index.name = "sha256"

# Col 0 is already the index (SHA256).
# Col 1 (iloc 0 after index_col) → malware type label.
# Cols 2+ (iloc 1+) → API function name columns, padded with empty strings.
label_col = df.iloc[:, 0]
api_cols  = df.iloc[:, 1:]

print(f"Rows loaded  : {len(df):,}")
print(f"Label counts : {label_col.value_counts().to_dict()}")
print(f"API columns  : {api_cols.shape[1]:,} (including empty padding)")

## Step 4 — Build Binary API Presence Matrix

Each row becomes a **set** of API names (empty strings and literal `"NA"` entries are excluded as padding artifacts). `MultiLabelBinarizer` converts these sets into a 0/1 matrix with one column per unique API name.

In [ ]:
EXCLUDED = {"NA", ""}

def row_to_api_set(row):
    """Return the set of real API names present in this row."""
    return {v for v in row.dropna().astype(str) if v not in EXCLUDED}

print("Parsing API sets per row …")
api_sets = api_cols.apply(row_to_api_set, axis=1)

print("Building binary API presence matrix …")
mlb = MultiLabelBinarizer(sparse_output=False)
api_matrix = pd.DataFrame(
    mlb.fit_transform(api_sets),
    index=df.index,
    columns=mlb.classes_,
    dtype="int8",
)

print(f"API matrix shape : {api_matrix.shape}  "
      f"({api_matrix.shape[0]:,} samples × {api_matrix.shape[1]:,} unique APIs)")
print(f"Memory usage     : {api_matrix.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 5 — One-Hot Encode Malware Labels

In [ ]:
label_matrix = pd.get_dummies(label_col.rename("label"), dtype="int8")
label_matrix.index = df.index

print(f"Label columns : {list(label_matrix.columns)}")
print(label_matrix.sum().rename("sample_count").to_string())

## Step 6 — Concatenate and Save

The final CSV has:
- `sha256` as the row index
- All unique API function name columns (0/1)
- All malware type columns (0/1) at the right-hand side

In [ ]:
result = pd.concat([api_matrix, label_matrix], axis=1)

print(f"Final dataset shape : {result.shape}  "
      f"({result.shape[0]:,} rows × {result.shape[1]:,} columns)")
print(f"Total memory        : {result.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nSaving to {OUTPUT_FILE} …")
result.to_csv(OUTPUT_FILE)
print("Done.")

## Step 7 — Download Result

Downloads `API_Functions_flat.csv` from the Colab runtime to your local machine.

> **Alternatively**, to save to Google Drive instead, mount Drive first and set `OUTPUT_FILE = "/content/drive/MyDrive/<your-folder>/API_Functions_flat.csv"` in Step 6.

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## (Optional) Preview the Result

In [ ]:
result.head(3)